# 📊 Sistema Profesional de Análisis de Portafolios — v3 Institucional
> **Módulo 6 | Finanzas Cuantitativas con Python**  
> Markowitz · Ledoit-Wolf · Black-Litterman · CVaR · Drawdown · Stress Testing · Factor Analysis · Rolling Analysis · Régimen de Mercado · Monte Carlo · Rebalanceo

---
**Flujo:** `1.Install` → `2.Imports` → `3.Inputs` → `4.Pipeline completo`

| Módulo | Descripción |
|---|---|
| A | Descarga y estadísticos | 
| B | Correlación + Ledoit-Wolf shrinkage |
| C | Markowitz · Black-Litterman · Risk Parity |
| D | Frontera Eficiente + CML |
| E | **CVaR · VaR histórico · Monte Carlo** |
| F | **Stress Testing (2008, 2020, 2022)** |
| G | **Max Drawdown · Calmar · Underwater** |
| H | **Rolling Sharpe/Volatilidad (1Y, 2Y, 3Y)** |
| I | **Análisis de Factores Fama-French** |
| J | **Régimen de Mercado (HMM proxy)** |
| K | **Monte Carlo forward-looking (1Y/3Y/5Y)** |
| L | **Trade List de Rebalanceo con costos** |
| M | Análisis Técnico |
| N | Dashboard HTML (tabs ampliados) |
| O | Excel (hojas ampliadas) |
| P | PDF institucional |

## 1. 📦 Instalación

In [ ]:
import subprocess, sys

paquetes = [
    'yfinance', 'pandas', 'numpy', 'scipy', 'plotly',
    'matplotlib', 'seaborn', 'openpyxl', 'fpdf2',
    'kaleido', 'ipywidgets', 'tqdm', 'scikit-learn', 'statsmodels'
]
for p in paquetes:
    subprocess.run([sys.executable,'-m','pip','install',p,'-q'], capture_output=True)

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

print('✅ Instalación completa.')

## 2. 📥 Importaciones

In [ ]:
import warnings, os
from datetime import datetime, timedelta
from itertools import product

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
from scipy.optimize import minimize
from scipy.stats import norm, skew, kurtosis
from sklearn.covariance import LedoitWolf
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import statsmodels.api as sm
from tqdm.notebook import tqdm

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

import ipywidgets as widgets
from IPython.display import display, clear_output

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from fpdf import FPDF

# ─── Tema dark financiero global ─────────────────────────────────────────────
pio.templates.default = 'plotly_dark'
PLOTLY_LAYOUT = dict(
    paper_bgcolor='#0d1117', plot_bgcolor='#0d1117',
    font=dict(family='Inter,Segoe UI,system-ui', color='#c9d1d9', size=12),
    title_font=dict(size=16, color='#58a6ff'),
    legend=dict(bgcolor='rgba(22,27,34,0.9)', bordercolor='#30363d',
                borderwidth=1, font=dict(size=11)),
    xaxis=dict(gridcolor='#21262d', zeroline=False, linecolor='#30363d'),
    yaxis=dict(gridcolor='#21262d', zeroline=False, linecolor='#30363d'),
    margin=dict(l=65, r=45, t=75, b=55)
)
COLORES = px.colors.qualitative.Bold
C_OK    = '#3fb950'
C_ERR   = '#ff7b72'
C_GOLD  = '#ffd700'
C_BLUE  = '#58a6ff'
C_WARN  = '#ffa657'

pd.set_option('display.float_format', '{:.4f}'.format)
print(f'✅ Imports OK | {datetime.today().strftime("%d/%m/%Y %H:%M")}')

## 3. 🎛️ Inputs Interactivos
> Configura y presiona **▶ EJECUTAR ANÁLISIS COMPLETO**

In [ ]:
PORTAFOLIO_ACTUAL = [
    'NU','VT','VOO','QQQM','GOOG','BRK-B',
    'NVO','NBIS','NVDA','AAPL','RDDT','MSFT',
    'PLTR','META','SLV'
]
UNIVERSO_EXTRA = [
    'AMZN','TSLA','JPM','V','MA','NFLX','AMD','INTC',
    'GLD','TLT','QQQ','SPY','IWM','XLF','XLE','XLK',
    'SNOW','CRM','UBER','COIN','TSM','ASML'
]
TODOS = sorted(set(PORTAFOLIO_ACTUAL + UNIVERSO_EXTRA))

# ── Tickers ───────────────────────────────────────────────────────────────────
w_tickers = widgets.SelectMultiple(
    options=TODOS, value=PORTAFOLIO_ACTUAL, rows=18,
    description='Tickers:', layout=widgets.Layout(width='260px'),
    style={'description_width':'60px'})

w_custom = widgets.Text(value='', placeholder='AMZN, BTC-USD ...',
    description='Agregar:', layout=widgets.Layout(width='300px'),
    style={'description_width':'60px'})
w_add = widgets.Button(description='+ Agregar', button_style='info',
    layout=widgets.Layout(width='100px', height='30px'))

def on_add(b):
    nuevos = [t.strip().upper() for t in w_custom.value.split(',') if t.strip()]
    if not nuevos: return
    opts = list(w_tickers.options); sel = list(w_tickers.value)
    for tk in nuevos:
        if tk not in opts: opts.append(tk)
        if tk not in sel:  sel.append(tk)
    w_tickers.options = sorted(opts)
    w_tickers.value   = [t for t in sel if t in w_tickers.options]
    w_custom.value    = ''
w_add.on_click(on_add)

# ── Parámetros base ───────────────────────────────────────────────────────────
w_periodo   = widgets.RadioButtons(
    options=[('Diaria (252)','diaria'),('Semanal (52)','semanal'),('Mensual (12)','mensual')],
    value='diaria', description='Período:', style={'description_width':'60px'})
w_rf        = widgets.FloatSlider(value=5.25, min=0, max=15, step=0.25,
    description='RF % anual:', readout_format='.2f',
    layout=widgets.Layout(width='360px'), style={'description_width':'100px'})
w_peso_min  = widgets.BoundedFloatText(value=5, min=0, max=20, step=1,
    description='Peso min %:', layout=widgets.Layout(width='175px'),
    style={'description_width':'85px'})
w_peso_max  = widgets.BoundedFloatText(value=25, min=5, max=100, step=5,
    description='Peso max %:', layout=widgets.Layout(width='175px'),
    style={'description_width':'85px'})

# ── Parámetros Black-Litterman ────────────────────────────────────────────────
w_bl_tau    = widgets.FloatSlider(value=0.05, min=0.01, max=0.5, step=0.01,
    description='BL tau:', readout_format='.2f',
    layout=widgets.Layout(width='360px'), style={'description_width':'100px'})
w_bl_views  = widgets.Textarea(
    value='NVDA:+0.05\nMETA:+0.03\nNVO:-0.02',
    placeholder='TICKER:view_anual_decimal\nEj: NVDA:+0.05',
    description='BL Views:', layout=widgets.Layout(width='340px', height='90px'),
    style={'description_width':'70px'})
w_bl_conf   = widgets.FloatSlider(value=50, min=10, max=100, step=10,
    description='BL conf %:', readout_format='.0f',
    layout=widgets.Layout(width='360px'), style={'description_width':'100px'})

# ── Stress test & Monte Carlo ─────────────────────────────────────────────────
w_mc_sims   = widgets.IntSlider(value=10000, min=1000, max=50000, step=1000,
    description='MC sims:', layout=widgets.Layout(width='360px'),
    style={'description_width':'100px'})
w_mc_años   = widgets.SelectMultiple(
    options=[1,3,5], value=[1,3,5], rows=3,
    description='MC años:', layout=widgets.Layout(width='175px'),
    style={'description_width':'70px'})

# ── Rebalanceo ────────────────────────────────────────────────────────────────
w_nav       = widgets.FloatText(value=137723.85, description='NAV USD:',
    layout=widgets.Layout(width='220px'), style={'description_width':'80px'})
w_comision  = widgets.FloatSlider(value=0.005, min=0, max=0.02, step=0.001,
    description='Comis %:', readout_format='.3f',
    layout=widgets.Layout(width='360px'), style={'description_width':'100px'})

# ── Botón principal ───────────────────────────────────────────────────────────
w_run    = widgets.Button(description='▶  EJECUTAR ANÁLISIS COMPLETO',
    button_style='success', layout=widgets.Layout(width='280px', height='44px'))
w_status = widgets.HTML(value='<span style="color:#8b949e">⏳ Configura y presiona ▶</span>')
out_area = widgets.Output()

# ── Layout ────────────────────────────────────────────────────────────────────
col_izq = widgets.VBox([
    widgets.HTML('<b style="color:#58a6ff">📋 Activos</b><br><small style="color:#8b949e">Ctrl+Click multiseleción</small>'),
    w_tickers, widgets.HBox([w_custom, w_add])])
col_der = widgets.VBox([
    widgets.HTML('<b style="color:#58a6ff">⚙️ Parámetros Base</b>'),
    w_periodo, w_rf, widgets.HBox([w_peso_min, w_peso_max]),
    widgets.HTML('<hr style="border-color:#30363d"><b style="color:#58a6ff">🔮 Black-Litterman</b>'),
    w_bl_views, w_bl_tau, w_bl_conf,
    widgets.HTML('<hr style="border-color:#30363d"><b style="color:#58a6ff">🎲 Monte Carlo + Rebalanceo</b>'),
    w_mc_sims, widgets.HBox([w_mc_años, w_nav]), w_comision,
    widgets.HTML('<hr style="border-color:#30363d">'),
    w_run, w_status
], layout=widgets.Layout(padding='0 0 0 28px'))

display(widgets.HBox([col_izq, col_der]))
display(out_area)
print('✅ Panel listo.')

## 4. 🚀 Pipeline Institucional Completo
> Todo el análisis corre al presionar **▶ EJECUTAR ANÁLISIS COMPLETO**

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# UTILIDADES GLOBALES
# ══════════════════════════════════════════════════════════════════════════════

def _t(texto):
    """Sanitiza texto a latin-1 para fpdf2."""
    mapa = {'\u2014':'--','\u2013':'-','\u2019':"'",'\u201c':'"','\u201d':'"',
            '\u2018':"'",'\u00e9':'e','\u00ed':'i','\u00f3':'o','\u00fa':'u',
            '\u00e1':'a','\u00f1':'n','\u00c1':'A','\u00c9':'E','\u00cd':'I',
            '\u00d3':'O','\u00da':'U','\u00d1':'N','\u00fc':'u','\u00e4':'a',
            '\u03bc':'u','\u03c3':'s','\u2265':'>=','\u2264':'<='}
    for k,v in mapa.items(): texto = str(texto).replace(k,v)
    return ''.join(c if ord(c)<256 else '?' for c in texto)

def apl(fig, titulo='', alto=500, **kw):
    """Aplica layout dark global a figura Plotly."""
    lay = dict(**PLOTLY_LAYOUT); lay['height']=alto
    if titulo: lay['title']=dict(text=titulo,font=dict(size=16,color=C_BLUE),x=0.02)
    lay.update(kw); fig.update_layout(**lay); return fig


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO A — DESCARGA Y ESTADÍSTICOS
# ══════════════════════════════════════════════════════════════════════════════

def descargar_precios(tickers, inicio, fin, intervalo):
    ok, mal = {}, []
    for tk in tqdm(tickers, desc='Descargando'):
        try:
            raw = yf.download(tk,start=inicio,end=fin,interval=intervalo,
                              progress=False,auto_adjust=False)
            if raw.empty or len(raw)<50:
                print(f'  ⚠ {tk}: insuficiente — descartado'); mal.append(tk); continue
            ok[tk] = raw['Close'].squeeze()
        except Exception as e:
            print(f'  ✗ {tk}: {e}'); mal.append(tk)
    if not ok: raise ValueError('Sin tickers válidos.')
    df = pd.DataFrame(ok); df.index=pd.to_datetime(df.index); df.index.name='Fecha'
    if df.isna().any().any(): df = df.ffill().bfill()
    print(f'✅ {len(df.columns)} activos | {len(df):,} obs | Excluidos: {mal or "ninguno"}')
    return df

def estadisticos_base(rend, factor, rf_p):
    mu_p=rend.mean(); std_p=rend.std(ddof=1)
    mu_a=mu_p*factor; std_a=std_p*np.sqrt(factor)
    rf_a=(1+rf_p)**factor-1
    sk=rend.apply(skew); ku=rend.apply(kurtosis)
    return pd.DataFrame({'Rend_Anual':mu_a,'Std_Anual':std_a,
                         'Sharpe':(mu_a-rf_a)/std_a,
                         'Skewness':sk,'Kurtosis':ku,
                         'Rend_Periodo':mu_p,'Std_Periodo':std_p})


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO B — COVARIANZA CON LEDOIT-WOLF SHRINKAGE
# ══════════════════════════════════════════════════════════════════════════════

def cov_ledoit_wolf(rend, factor):
    """
    Estimación robusta de covarianza via Ledoit-Wolf shrinkage.
    Reduce el error de estimación de la matriz muestral.
    Retorna cov anualizada y coeficiente de shrinkage aplicado.
    """
    lw = LedoitWolf().fit(rend.values)
    cov_lw = pd.DataFrame(lw.covariance_*factor,
                          index=rend.columns, columns=rend.columns)
    return cov_lw, lw.shrinkage_


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO C — OPTIMIZADORES
# ══════════════════════════════════════════════════════════════════════════════

def optimizar(mu, Sigma, rf, N, w_min, w_max, obj='sharpe'):
    """Optimizador SLSQP genérico: obj = 'sharpe' | 'varianza' | 'risk_parity'."""
    bounds = [(w_min, w_max)]*N
    cons   = [{'type':'eq','fun':lambda w: np.sum(w)-1.0}]
    w0     = np.full(N, 1/N)
    if obj=='varianza':
        fun = lambda w: float(w@Sigma@w)
    elif obj=='risk_parity':
        def fun(w):
            var  = float(w@Sigma@w)
            mrc  = Sigma@w            # Marginal Risk Contribution
            rc   = w*mrc/np.sqrt(var) # Risk Contribution
            rc_t = np.sqrt(var)/N     # Target: igual contribución
            return np.sum((rc-rc_t)**2)
    else:  # sharpe
        fun = lambda w: -(float(w@mu)-rf)/np.sqrt(float(w@Sigma@w))
    res = minimize(fun,w0,method='SLSQP',bounds=bounds,constraints=cons,
                   options={'maxiter':3000,'ftol':1e-13})
    w = res.x; r=float(w@mu); s=np.sqrt(float(w@Sigma@w))
    return {'pesos':w,'rendimiento':r,'riesgo':s,'sharpe':(r-rf)/s,'conv':res.success}


def black_litterman(mu_eq, Sigma, rf, tickers, views_str, tau, conf_pct):
    """
    Black-Litterman: combina retornos de equilibrio con views del PM.

    Parámetros
    ----------
    mu_eq    : retornos implícitos de equilibrio (CAPM proxy: Sigma@w_mkt * lambda_)
    Sigma    : matriz de covarianza anualizada
    views_str: string 'TICKER:view\nTICKER2:view2'
    tau      : escalar de incertidumbre (típico 0.01-0.10)
    conf_pct : confianza en los views (10-100%)
    """
    n = len(tickers)
    # Parsear views
    P_rows, Q_vals = [], []
    for line in views_str.strip().split('\n'):
        line = line.strip()
        if ':' not in line: continue
        tk, val = line.split(':',1)
        tk = tk.strip().upper()
        if tk not in tickers: continue
        p = np.zeros(n)
        p[tickers.index(tk)] = 1.0
        P_rows.append(p)
        Q_vals.append(float(val))
    if not P_rows:
        print('  BL: sin views válidos — usando equilibrio puro')
        return mu_eq
    P = np.array(P_rows)
    Q = np.array(Q_vals)
    # Omega: matriz de incertidumbre de views (proporcional a Sigma)
    omega_scale = 1 / (conf_pct/100 + 1e-10)
    Omega = np.diag(np.diag(P @ (tau*Sigma) @ P.T)) * omega_scale
    # Fórmula BL
    tSigma = tau * Sigma
    M1 = np.linalg.inv(tSigma)
    M2 = P.T @ np.linalg.inv(Omega) @ P
    mu_bl = np.linalg.inv(M1+M2) @ (M1@mu_eq + P.T@np.linalg.inv(Omega)@Q)
    return mu_bl


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO D — FRONTERA EFICIENTE
# ══════════════════════════════════════════════════════════════════════════════

def simular_frontera(mu, Sigma, rf, N, w_min, w_max, n_sim=500):
    sr, ss, sh, sw = [],[],[],[]
    pb = tqdm(total=n_sim, desc='Frontera')
    while len(sr)<n_sim:
        w = np.clip(np.random.dirichlet(np.ones(N)), w_min, w_max)
        w /= w.sum()
        r=float(w@mu); s=np.sqrt(float(w@Sigma@w))
        sr.append(r); ss.append(s); sh.append((r-rf)/s); sw.append(w); pb.update(1)
    pb.close()
    df = pd.DataFrame({'Rendimiento':sr,'Riesgo':ss,'Sharpe':sh})
    return df, sw


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO E — RIESGO AVANZADO: CVaR, VaR HISTÓRICO, MONTE CARLO
# ══════════════════════════════════════════════════════════════════════════════

def calcular_riesgo_completo(rend_port_diario, mu_a, sig_a, rf_a, label):
    """
    Calcula el stack completo de métricas de riesgo para un portafolio.

    Incluye: VaR paramétrico, VaR histórico, CVaR (Expected Shortfall),
    VaR Monte Carlo, todos al 95% y 99%.
    """
    Z95, Z99 = norm.ppf(0.95), norm.ppf(0.99)
    mu_d  = mu_a/252; sig_d = sig_a/np.sqrt(252)

    # VaR paramétrico
    var95_p = -(mu_d - Z95*sig_d)
    var99_p = -(mu_d - Z99*sig_d)

    # VaR histórico (percentil de pérdidas reales)
    r = rend_port_diario.dropna()
    var95_h = float(-np.percentile(r, 5))
    var99_h = float(-np.percentile(r, 1))

    # CVaR / Expected Shortfall (pérdida esperada DADO que supera el VaR)
    cvar95 = float(-r[r <= -var95_h].mean()) if (-r<=var95_h*-1).any() else var95_h
    cvar99 = float(-r[r <= -var99_h].mean()) if (-r<=var99_h*-1).any() else var99_h
    # Forma correcta
    thresh95 = np.percentile(r, 5)
    thresh99 = np.percentile(r, 1)
    cvar95   = float(-r[r<=thresh95].mean())
    cvar99   = float(-r[r<=thresh99].mean())

    # VaR Monte Carlo (10,000 simulaciones diarias)
    mc_r = np.random.normal(mu_d, sig_d, 100000)
    var95_mc = float(-np.percentile(mc_r, 5))
    var99_mc = float(-np.percentile(mc_r, 1))

    return {
        'Portafolio':     label,
        'VaR 95% Param':  var95_p,
        'VaR 99% Param':  var99_p,
        'VaR 95% Hist':   var95_h,
        'VaR 99% Hist':   var99_h,
        'CVaR 95% (ES)':  cvar95,
        'CVaR 99% (ES)':  cvar99,
        'VaR 95% MC':     var95_mc,
        'VaR 99% MC':     var99_mc,
        'VaR 99% Mensual': -(mu_d*21 - Z99*sig_d*np.sqrt(21)),
        'VaR 99% Anual':  -(mu_a - Z99*sig_a),
    }


def fig_var_distribucion(rend_port, label):
    """Histograma de retornos con VaR y CVaR anotados."""
    r = rend_port.dropna()
    var95 = np.percentile(r, 5)
    var99 = np.percentile(r, 1)
    cvar99 = r[r<=var99].mean()

    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=r, nbinsx=80, name='Retornos',
        marker_color=C_BLUE, opacity=0.7,
        histnorm='probability density'
    ))
    # Curva normal teórica
    x_n = np.linspace(r.min(), r.max(), 300)
    y_n = norm.pdf(x_n, r.mean(), r.std())
    fig.add_trace(go.Scatter(x=x_n,y=y_n,name='Normal teórica',
                             line=dict(color=C_WARN,width=2)))
    # Líneas VaR
    for xv,col,nm in [(var95,C_ERR,'VaR 95%'),(var99,'#ff0000','VaR 99%'),(cvar99,C_GOLD,'CVaR 99%')]:
        fig.add_vline(x=xv, line_dash='dash', line_color=col,
                      annotation_text=nm, annotation_position='top right',
                      annotation_font_color=col)
    fig.update_xaxes(title='Retorno diario', tickformat='.1%')
    fig.update_yaxes(title='Densidad')
    return apl(fig, f'Distribución de Retornos — {label} | Fat Tails vs Normal', alto=420)


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO F — STRESS TESTING HISTÓRICO
# ══════════════════════════════════════════════════════════════════════════════

PERIODOS_STRESS = {
    'Crisis 2008 (Lehman)':        ('2008-09-01', '2009-03-31'),
    'Flash Crash 2010':            ('2010-04-23', '2010-07-02'),
    'Taper Tantrum 2013':          ('2013-05-22', '2013-06-24'),
    'Caída petróleo 2015-16':      ('2015-08-01', '2016-02-29'),
    'COVID Crash 2020':            ('2020-02-19', '2020-03-23'),
    'Alza de tasas 2022':          ('2022-01-03', '2022-10-14'),
    'Crisis bancaria 2023':        ('2023-03-08', '2023-03-31'),
}

def stress_testing(precios_diarios, pesos_pmv, pesos_pms, tickers):
    """
    Simula el comportamiento del portafolio en períodos de estrés histórico.
    Usa precios diarios descargados (columnas = tickers).
    """
    resultados = []
    for nombre, (ini, fin) in PERIODOS_STRESS.items():
        sub = precios_diarios.loc[ini:fin, [t for t in tickers if t in precios_diarios.columns]]
        if len(sub) < 5: continue
        rend_sub = np.log(sub/sub.shift(1)).dropna()
        tks = sub.columns.tolist()
        for pesos_arr, port_lbl in [(pesos_pmv,'PMV'),(pesos_pms,'PMS')]:
            w = np.array([pesos_arr[tickers.index(t)] if t in tickers else 0 for t in tks])
            if w.sum()>0: w/=w.sum()
            rp = (rend_sub*w).sum(axis=1)
            retorno_total  = float(np.exp(rp.sum())-1)
            max_dd         = float((np.exp(rp.cumsum()) - np.exp(rp.cumsum().cummax())).min())
            vol_anual      = float(rp.std()*np.sqrt(252))
            resultados.append({
                'Período':        nombre,
                'Portafolio':     port_lbl,
                'Retorno Total':  retorno_total,
                'Max Drawdown':   max_dd,
                'Vol Anual':      vol_anual,
                'Días':           len(rp)
            })
    return pd.DataFrame(resultados).set_index(['Período','Portafolio'])


def fig_stress(df_stress):
    """Heatmap de stress testing: retornos por período y portafolio."""
    pivot = df_stress['Retorno Total'].unstack('Portafolio')
    txt   = [[f'{v:.1%}' for v in row] for row in pivot.values]
    fig   = go.Figure(go.Heatmap(
        z=pivot.values, x=pivot.columns.tolist(), y=pivot.index.tolist(),
        text=txt, texttemplate='%{text}', textfont=dict(size=12),
        colorscale='RdYlGn', zmid=0,
        colorbar=dict(title='Retorno',tickformat='.0%',thickness=14,
                      tickfont=dict(color='#c9d1d9'))
    ))
    return apl(fig,'Stress Testing — Retorno Total por Período de Crisis', alto=380)


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO G — DRAWDOWN ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

def calcular_drawdown(rend_serie):
    """
    Retorna serie de drawdown, max drawdown, Calmar ratio y tiempo de recuperación.
    """
    cumret  = np.exp(rend_serie.cumsum())
    peak    = cumret.cummax()
    dd      = (cumret - peak) / peak        # Drawdown serie
    max_dd  = float(dd.min())

    # Tiempo de recuperación (días desde máx drawdown hasta siguiente nuevo máx)
    idx_trough = dd.idxmin()
    after_trough = cumret.loc[idx_trough:]
    peak_before  = float(peak.loc[idx_trough])
    recovered    = after_trough[after_trough >= peak_before]
    recovery_days = (recovered.index[0] - idx_trough).days if len(recovered)>0 else None

    # Calmar ratio: retorno anual / abs(max drawdown)
    ret_anual = float(rend_serie.mean()*252)
    calmar    = ret_anual / abs(max_dd) if max_dd!=0 else np.nan

    # Underwater plot data
    return dd, max_dd, calmar, recovery_days


def fig_drawdown(rend_pmv, rend_pms, tickers_rend=None, tickers=None):
    """Underwater chart con PMV, PMS y activos individuales (opcional)."""
    fig = go.Figure()
    for rp, lbl, col in [(rend_pmv,'PMV',C_ERR),(rend_pms,'PMS',C_GOLD)]:
        dd,_,_,_ = calcular_drawdown(rp)
        fig.add_trace(go.Scatter(
            x=dd.index, y=dd, name=lbl, fill='tozeroy',
            line=dict(color=col,width=1.5),
            fillcolor=col.replace('#','rgba(').replace(')',',0.15)') if col.startswith('#') else col,
            hovertemplate=f'{lbl}: %{{y:.2%}}'
        ))
    if tickers_rend is not None and tickers is not None:
        for i,tk in enumerate(tickers[:5]):  # Top 5 para no saturar
            dd,_,_,_ = calcular_drawdown(tickers_rend[tk])
            fig.add_trace(go.Scatter(
                x=dd.index, y=dd, name=tk, line=dict(color=COLORES[i],width=1,dash='dot'),
                opacity=0.5, hovertemplate=f'{tk}: %{{y:.2%}}'
            ))
    fig.update_yaxes(title='Drawdown', tickformat='.0%')
    fig.update_xaxes(title='Fecha')
    return apl(fig,'Underwater Chart — Drawdown Histórico', alto=430)


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO H — ROLLING ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

def rolling_metrics(rend_port, factor, rf_a, ventanas_dias=[252, 504, 756]):
    """
    Calcula Sharpe, volatilidad y retorno rolling en múltiples ventanas.
    Ventanas: 252d (1Y), 504d (2Y), 756d (3Y).
    """
    resultados = {}
    rf_d = (1+rf_a)**(1/factor)-1
    for v in ventanas_dias:
        mu_r  = rend_port.rolling(v).mean() * factor
        sig_r = rend_port.rolling(v).std(ddof=1) * np.sqrt(factor)
        sh_r  = (mu_r - rf_a) / sig_r
        resultados[f'{v//252}Y'] = {'sharpe':sh_r, 'vol':sig_r, 'rend':mu_r}
    return resultados


def fig_rolling(rolling_pmv, rolling_pms, metrica='sharpe'):
    """Gráfica de métricas rolling con subplots por ventana."""
    ventanas = list(rolling_pmv.keys())
    fig = make_subplots(rows=len(ventanas), cols=1,
                        shared_xaxes=True, vertical_spacing=0.05,
                        subplot_titles=[f'Ventana {v}' for v in ventanas])
    titulos = {'sharpe':'Sharpe Ratio Rolling','vol':'Volatilidad Rolling (anual)',
               'rend':'Retorno Rolling (anual)'}
    for i, v in enumerate(ventanas):
        for data, lbl, col in [(rolling_pmv[v],'PMV',C_ERR),(rolling_pms[v],'PMS',C_GOLD)]:
            serie = data[metrica]
            fig.add_trace(go.Scatter(
                x=serie.index, y=serie,
                name=f'{lbl} {v}', line=dict(color=col,width=1.5),
                hovertemplate=f'{lbl}: %{{y:.3f}}'
            ), row=i+1, col=1)
        if metrica in ('vol','rend'):
            fig.update_yaxes(tickformat='.0%', row=i+1, col=1)
        fig.update_xaxes(gridcolor='#21262d', row=i+1, col=1)
        fig.update_yaxes(gridcolor='#21262d', row=i+1, col=1)
    return apl(fig, titulos.get(metrica,'Rolling'), alto=550)


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO I — ANÁLISIS DE FACTORES (FAMA-FRENCH 3F PROXY)
# ══════════════════════════════════════════════════════════════════════════════

def descargar_factores_ff(inicio, fin):
    """
    Construye factores Fama-French proxy desde ETFs públicos:
    Mkt-Rf: SPY-RF | SMB: IWM-SPY | HML: IVE-IWF | MOM: MTUM-USMV
    """
    etfs = ['SPY','IWM','IVE','IWF','MTUM','USMV','BIL']
    raw = {}
    for tk in tqdm(etfs, desc='Factores FF'):
        try:
            d = yf.download(tk,start=inicio,end=fin,interval='1d',
                            progress=False,auto_adjust=False)
            if not d.empty: raw[tk] = np.log(d['Close'].squeeze()/d['Close'].squeeze().shift(1))
        except: pass
    if 'SPY' not in raw: return None
    rf_d = raw.get('BIL', pd.Series(0, index=raw['SPY'].index))
    mkt  = raw['SPY'] - rf_d
    smb  = raw.get('IWM', raw['SPY']) - raw['SPY']   # Small minus Big
    hml  = raw.get('IVE', raw['SPY']) - raw.get('IWF', raw['SPY'])  # Value minus Growth
    mom  = raw.get('MTUM', raw['SPY']) - raw.get('USMV', raw['SPY'])  # Momentum
    df   = pd.DataFrame({'Mkt_RF':mkt,'SMB':smb,'HML':hml,'MOM':mom}).dropna()
    return df


def regresion_factores(rend_port, factores_df):
    """
    Regresión OLS del portafolio contra factores FF.
    Retorna DataFrame con betas, R2, alpha anual y t-stats.
    """
    data = rend_port.to_frame('Portfolio').join(factores_df).dropna()
    Y    = data['Portfolio']
    X    = sm.add_constant(data[factores_df.columns])
    res  = sm.OLS(Y, X).fit()
    out  = pd.DataFrame({
        'Beta':    res.params,
        't-stat':  res.tvalues,
        'p-value': res.pvalues
    })
    out.index.name = 'Factor'
    alpha_anual = res.params.get('const', 0) * 252
    return out, res.rsquared, alpha_anual


def fig_factor_betas(betas_pmv, betas_pms):
    """Gráfica comparativa de betas de factores PMV vs PMS."""
    factores = [i for i in betas_pmv.index if i!='const']
    fig = go.Figure()
    for betas, lbl, col in [(betas_pmv,'PMV',C_ERR),(betas_pms,'PMS',C_GOLD)]:
        vals = [betas.loc[f,'Beta'] for f in factores if f in betas.index]
        fig.add_trace(go.Bar(name=lbl, x=factores, y=vals,
                             marker_color=col, opacity=0.85,
                             text=[f'{v:.3f}' for v in vals],
                             textposition='outside'))
    fig.add_hline(y=0, line_color='#484f58')
    fig.update_yaxes(title='Beta')
    return apl(fig,'Exposición a Factores Fama-French — PMV vs PMS', alto=400, barmode='group')


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO J — RÉGIMEN DE MERCADO (K-MEANS PROXY)
# ══════════════════════════════════════════════════════════════════════════════

def detectar_regimenes(rend_port, n_regimenes=3):
    """
    Clasifica el mercado en regímenes (Bull / Bear / Lateral) usando
    K-Means sobre retorno rolling y volatilidad rolling.
    Proxy simplificado del Hidden Markov Model (HMM).
    """
    r = rend_port.dropna()
    df = pd.DataFrame({
        'ret_21':  r.rolling(21).mean() * 252,
        'vol_21':  r.rolling(21).std() * np.sqrt(252),
        'ret_63':  r.rolling(63).mean() * 252,
    }).dropna()

    scaler  = StandardScaler()
    X_scaled = scaler.fit_transform(df)
    km      = KMeans(n_clusters=n_regimenes, random_state=42, n_init=10)
    labels  = km.fit_predict(X_scaled)
    df['Regimen'] = labels

    # Ordenar regímenes por retorno medio (0=Bear, 1=Lateral, 2=Bull)
    medias = df.groupby('Regimen')['ret_21'].mean().sort_values()
    mapa   = {old:new for new,old in enumerate(medias.index)}
    df['Regimen'] = df['Regimen'].map(mapa)
    nombres = {0:'Bear 🔴', 1:'Lateral ⚪', 2:'Bull 🟢'}
    df['Nombre'] = df['Regimen'].map(nombres)

    # Métricas por régimen
    stats_reg = df.groupby('Nombre').agg(
        Retorno_Anual=('ret_21','mean'),
        Volatilidad=('vol_21','mean'),
        Dias=('ret_21','count')
    ).round(4)

    return df, stats_reg


def fig_regimenes(precios_port, regimenes_df):
    """Gráfica de precio con fondo coloreado por régimen."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=precios_port.index, y=precios_port,
        line=dict(color=C_BLUE,width=1.5), name='Portafolio'
    ))
    colores_reg = {0:'rgba(255,123,114,0.12)', 1:'rgba(139,148,158,0.10)', 2:'rgba(63,185,80,0.12)'}
    nombres_reg = {0:'Bear 🔴', 1:'Lateral ⚪', 2:'Bull 🟢'}
    prev_reg, prev_date = None, None
    for date, row in regimenes_df.iterrows():
        reg = row['Regimen']
        if prev_reg is None:
            prev_reg, prev_date = reg, date; continue
        if reg != prev_reg or date == regimenes_df.index[-1]:
            fig.add_vrect(
                x0=prev_date, x1=date,
                fillcolor=colores_reg.get(prev_reg,'rgba(0,0,0,0.05)'),
                line_width=0,
                annotation_text=nombres_reg.get(prev_reg,''),
                annotation_position='top left',
                annotation_font_size=9
            )
            prev_reg, prev_date = reg, date
    fig.update_yaxes(title='Valor (base 100)')
    fig.update_xaxes(title='Fecha')
    return apl(fig,'Regímenes de Mercado — Bull / Bear / Lateral', alto=460)


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO K — MONTE CARLO FORWARD-LOOKING
# ══════════════════════════════════════════════════════════════════════════════

def monte_carlo_forward(mu_a, sig_a, nav, n_sims, años_list, label):
    """
    Simulación GBM (Geometric Brownian Motion) forward-looking.
    Genera distribución de outcomes para cada horizonte temporal.
    Retorna percentiles clave: 5, 25, 50, 75, 95.
    """
    mu_d  = mu_a / 252
    sig_d = sig_a / np.sqrt(252)
    resultados = {}

    for años in años_list:
        dias = int(años * 252)
        # GBM: S_t = S_0 * exp((mu - 0.5*sig^2)*t + sig*sqrt(t)*Z)
        shocks = np.random.normal(0, 1, (n_sims, dias))
        log_ret = (mu_d - 0.5*sig_d**2) + sig_d*shocks
        cum_ret = np.exp(log_ret.cumsum(axis=1))
        final   = cum_ret[:, -1] * nav

        resultados[años] = {
            'paths_sample': cum_ret[:100]*nav,  # 100 trayectorias para graficar
            'final': final,
            'p5':   float(np.percentile(final, 5)),
            'p25':  float(np.percentile(final, 25)),
            'p50':  float(np.percentile(final, 50)),
            'p75':  float(np.percentile(final, 75)),
            'p95':  float(np.percentile(final, 95)),
            'prob_positivo': float(np.mean(final > nav)),
            'prob_2x':       float(np.mean(final > 2*nav)),
        }
    return resultados


def fig_monte_carlo(mc_results, nav, años_list, label):
    """Gráficas de Monte Carlo: trayectorias + distribución final por horizonte."""
    fig = make_subplots(
        rows=len(años_list), cols=2,
        subplot_titles=[
            item for a in años_list
            for item in [f'Trayectorias {a}Y — {label}', f'Distribución Final {a}Y']
        ],
        horizontal_spacing=0.08, vertical_spacing=0.12
    )
    for i, años in enumerate(años_list):
        res = mc_results[años]
        dias = int(años*252)
        eje_x = list(range(dias+1))

        # Trayectorias (sample)
        for j, path in enumerate(res['paths_sample'][:50]):
            fig.add_trace(go.Scatter(
                x=eje_x[1:], y=path,
                line=dict(color=C_BLUE,width=0.4), opacity=0.3,
                showlegend=False
            ), row=i+1, col=1)

        # Percentiles
        for pct, col_p, nm in [('p95',C_OK,'p95'),('p75',C_WARN,'p75'),
                                ('p50',C_BLUE,'Mediana'),
                                ('p25',C_ERR,'p25'),('p5','#ff0000','p5')]:
            val_f = res[pct]
            # Solo dibujar como línea horizontal en el punto final
            fig.add_trace(go.Scatter(
                x=[1,dias], y=[nav, val_f],
                line=dict(color=col_p,width=1.8,dash='dash'),
                name=f'{nm}: ${val_f:,.0f}', showlegend=(i==0)
            ), row=i+1, col=1)
        fig.add_hline(y=nav, line_dash='dot', line_color='#484f58', row=i+1, col=1)

        # Histograma distribución final
        fig.add_trace(go.Histogram(
            x=res['final'], nbinsx=60,
            marker_color=C_BLUE, opacity=0.7, showlegend=False,
            name=f'Dist {años}Y'
        ), row=i+1, col=2)
        fig.add_vline(x=nav, line_dash='dash', line_color=C_WARN, row=i+1, col=2)
        fig.add_vline(x=res['p50'], line_dash='dash', line_color=C_OK, row=i+1, col=2)

        fig.update_xaxes(gridcolor='#21262d', row=i+1, col=1)
        fig.update_xaxes(gridcolor='#21262d', title='USD Final', row=i+1, col=2)
        fig.update_yaxes(gridcolor='#21262d', title='USD', row=i+1, col=1)
        fig.update_yaxes(gridcolor='#21262d', row=i+1, col=2)

    return apl(fig, f'Monte Carlo GBM — Proyección Forward-Looking | {label}',
               alto=380*len(años_list))


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO L — TRADE LIST DE REBALANCEO
# ══════════════════════════════════════════════════════════════════════════════

def calcular_rebalanceo(tickers, pesos_objetivo, precios_actuales,
                        nav, comision_pct, posiciones_actuales=None):
    """
    Genera trade list de rebalanceo desde posición actual a pesos objetivo.

    posiciones_actuales: dict {ticker: n_acciones} — si None, asume 0 (posición nueva)
    Retorna DataFrame con operaciones, costo estimado y turnover.
    """
    if posiciones_actuales is None:
        posiciones_actuales = {tk: 0 for tk in tickers}

    trades = []
    for tk, w_obj in zip(tickers, pesos_objetivo):
        precio  = precios_actuales.get(tk, np.nan)
        if np.isnan(precio): continue
        usd_obj = nav * w_obj
        acc_obj = usd_obj / precio
        acc_act = posiciones_actuales.get(tk, 0)
        delta   = acc_obj - acc_act
        usd_trade = abs(delta) * precio
        costo   = usd_trade * comision_pct
        w_act   = (acc_act * precio) / nav if nav > 0 else 0
        trades.append({
            'Ticker':         tk,
            'Peso Actual':    w_act,
            'Peso Objetivo':  w_obj,
            'Gap':            w_obj - w_act,
            'Precio':         precio,
            'Acc. Actuales':  acc_act,
            'Acc. Objetivo':  round(acc_obj, 2),
            'Delta Acciones': round(delta, 2),
            'USD Trade':      round(usd_trade, 2),
            'Dirección':      'COMPRAR' if delta>0.5 else ('VENDER' if delta<-0.5 else 'MANTENER'),
            'Comisión Est.':  round(costo, 2),
        })

    df = pd.DataFrame(trades).set_index('Ticker')
    df['Turnover Total USD'] = df['USD Trade'].sum()
    df['Costo Total Est.']   = df['Comisión Est.'].sum()
    df['Turnover %']         = df['USD Trade'].sum() / nav
    return df


def fig_rebalanceo(df_reb, titulo='Rebalanceo hacia PMS'):
    """Gráfica de waterfall: gap de pesos actual vs objetivo."""
    df = df_reb[df_reb['Dirección']!='MANTENER'].copy()
    colores = [C_OK if g>0 else C_ERR for g in df['Gap']]
    fig = go.Figure(go.Bar(
        x=df.index, y=df['Gap'],
        marker_color=colores,
        text=[f'{v:.2%}' for v in df['Gap']], textposition='outside',
        hovertemplate='%{x}<br>Gap: %{y:.2%}<br>USD: $%{customdata:,.0f}',
        customdata=df['USD Trade']
    ))
    fig.add_hline(y=0, line_color='#484f58')
    fig.update_yaxes(title='Diferencia de Peso', tickformat='.0%')
    return apl(fig, titulo, alto=380)


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO M — ANÁLISIS TÉCNICO
# ══════════════════════════════════════════════════════════════════════════════

def calcular_rsi(c, p=14):
    d=c.diff(); g=d.clip(lower=0).ewm(alpha=1/p,adjust=False).mean()
    l=(-d.clip(upper=0)).ewm(alpha=1/p,adjust=False).mean()
    return 100-100/(1+g/l.replace(0,np.nan))

def calcular_macd(c, f=12, s=26, sig=9):
    m=c.ewm(span=f,adjust=False).mean()-c.ewm(span=s,adjust=False).mean()
    sg=m.ewm(span=sig,adjust=False).mean(); return m,sg,m-sg

def calcular_estocastico(c, k=14, d=3, sm=3):
    lo=c.rolling(k).min(); hi=c.rolling(k).max()
    pk=((c-lo)/(hi-lo+1e-10)*100).rolling(sm).mean()
    return pk,pk.rolling(d).mean()

def detectar_señales(c,rsi,ml,ms):
    out=[]
    s50,s200=c.rolling(50).mean(),c.rolling(200).mean()
    if len(s50.dropna())>2 and len(s200.dropna())>2:
        if s50.iloc[-2]<s200.iloc[-2] and s50.iloc[-1]>=s200.iloc[-1]: out.append('Golden Cross (SMA50>SMA200)')
        elif s50.iloc[-2]>s200.iloc[-2] and s50.iloc[-1]<=s200.iloc[-1]: out.append('Death Cross (SMA50<SMA200)')
    r=rsi.iloc[-1]
    if r>=70: out.append(f'RSI sobrecomprado: {r:.1f}')
    elif r<=30: out.append(f'RSI sobrevendido: {r:.1f}')
    if ml.iloc[-2]<ms.iloc[-2] and ml.iloc[-1]>=ms.iloc[-1]: out.append('MACD cruce alcista')
    elif ml.iloc[-2]>ms.iloc[-2] and ml.iloc[-1]<=ms.iloc[-1]: out.append('MACD cruce bajista')
    return out or ['Sin señales destacadas (90 dias)']

def fig_tecnica(tk, ind):
    fig=make_subplots(rows=5,cols=1,shared_xaxes=True,
                      row_heights=[0.38,0.14,0.16,0.16,0.16],
                      vertical_spacing=0.025,
                      subplot_titles=[f'{tk} Precio+SMAs','Volumen','RSI(14)','MACD(12,26,9)','Estocastico(14,3,3)'])
    idx=ind['close'].index
    fig.add_trace(go.Scatter(x=idx,y=ind['close'],name='Close',line=dict(color=C_BLUE,width=1.8)),row=1,col=1)
    for k,col,nm in [('sma50','#f0e130','SMA50'),('sma100',C_OK,'SMA100'),('sma200',C_ERR,'SMA200')]:
        fig.add_trace(go.Scatter(x=idx,y=ind[k],name=nm,line=dict(color=col,width=1.2,dash='dot')),row=1,col=1)
    if not ind['volume'].empty:
        vc=['#3fb950' if d>=0 else '#ff7b72' for d in ind['close'].diff().fillna(0)]
        fig.add_trace(go.Bar(x=idx,y=ind['volume'],name='Vol',marker_color=vc,opacity=0.7),row=2,col=1)
    fig.add_trace(go.Scatter(x=idx,y=ind['rsi'],name='RSI',line=dict(color='#bc8cff',width=1.5)),row=3,col=1)
    for lv,c in [(70,'rgba(255,123,114,0.5)'),(30,'rgba(63,185,80,0.5)')]:
        fig.add_hline(y=lv,line_dash='dash',line_color=c,row=3,col=1)
    fig.add_hrect(y0=70,y1=100,fillcolor='rgba(255,123,114,0.06)',row=3,col=1)
    fig.add_hrect(y0=0,y1=30,fillcolor='rgba(63,185,80,0.06)',row=3,col=1)
    hc=['#3fb950' if v>=0 else '#ff7b72' for v in ind['macd_h'].fillna(0)]
    fig.add_trace(go.Bar(x=idx,y=ind['macd_h'],name='Histo',marker_color=hc,opacity=0.7),row=4,col=1)
    fig.add_trace(go.Scatter(x=idx,y=ind['macd'],name='MACD',line=dict(color=C_BLUE,width=1.5)),row=4,col=1)
    fig.add_trace(go.Scatter(x=idx,y=ind['macd_s'],name='Señal',line=dict(color=C_WARN,width=1.5)),row=4,col=1)
    fig.add_trace(go.Scatter(x=idx,y=ind['pct_k'],name='%K',line=dict(color='#79c0ff',width=1.5)),row=5,col=1)
    fig.add_trace(go.Scatter(x=idx,y=ind['pct_d'],name='%D',line=dict(color=C_WARN,width=1.5,dash='dot')),row=5,col=1)
    for lv in [80,20]:
        fig.add_hline(y=lv,line_dash='dash',line_color='rgba(255,123,114,0.5)' if lv==80 else 'rgba(63,185,80,0.5)',row=5,col=1)
    fig.update_layout(height=920,**{k:v for k,v in PLOTLY_LAYOUT.items() if k not in ('xaxis','yaxis','margin')},
                      margin=dict(l=60,r=30,t=70,b=40),showlegend=True,
                      legend=dict(orientation='h',y=-0.03,font=dict(size=10),bgcolor='rgba(22,27,34,0.9)'))
    for i in range(1,6):
        fig.update_xaxes(gridcolor='#21262d',linecolor='#30363d',row=i,col=1)
        fig.update_yaxes(gridcolor='#21262d',linecolor='#30363d',row=i,col=1)
    return fig

def fig_base100(inds, tickers):
    fig=go.Figure()
    for i,tk in enumerate(tickers):
        c=inds[tk]['close'].dropna(); n=(c/c.iloc[0])*100
        fig.add_trace(go.Scatter(x=c.index,y=n,name=tk,
                                 line=dict(color=COLORES[i%len(COLORES)],width=1.8)))
    fig.add_hline(y=100,line_dash='dash',line_color='#484f58')
    fig.update_xaxes(title='Fecha'); fig.update_yaxes(title='Indice base 100')
    return apl(fig,'Rendimiento Comparativo — Base 100',alto=440)


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO N — HEATMAP CORRELACIÓN
# ══════════════════════════════════════════════════════════════════════════════

def fig_heatmap_corr(corr):
    n=len(corr); txt=[[f'{v:.2f}' for v in row] for row in corr.values]
    fig=go.Figure(go.Heatmap(
        z=corr.values,x=corr.columns.tolist(),y=corr.index.tolist(),
        text=txt,texttemplate='%{text}',textfont=dict(size=max(7,11-n//3)),
        colorscale=[[0,'#d73027'],[0.25,'#f46d43'],[0.5,'#1a1a2e'],
                    [0.75,'#4575b4'],[1,'#313695']],
        zmid=0,zmin=-1,zmax=1,
        colorbar=dict(title='rho',thickness=14,len=0.8,tickfont=dict(color='#c9d1d9'))))
    return apl(fig,'Matriz de Correlacion — Rendimientos Logaritmicos',alto=max(420,48*n))


def fig_frontera(df_sim,pmv,pms,stats,tickers,rf_a):
    fig=go.Figure()
    fig.add_trace(go.Scatter(
        x=df_sim['Riesgo'],y=df_sim['Rendimiento'],mode='markers',
        marker=dict(color=df_sim['Sharpe'],colorscale='Viridis',size=5,opacity=0.6,
                    colorbar=dict(title='Sharpe',thickness=12,tickfont=dict(color='#c9d1d9'))),
        hovertemplate='sigma:%{x:.2%}<br>mu:%{y:.2%}<br>Sharpe:%{marker.color:.3f}',
        name='Simulados'))
    sig_r=np.linspace(0,df_sim['Riesgo'].max()*1.1,200)
    fig.add_trace(go.Scatter(x=sig_r,y=rf_a+pms['sharpe']*sig_r,
                             mode='lines',line=dict(color=C_BLUE,width=2,dash='dot'),name='CML'))
    for pt,lbl,col in [(pmv,'PMV',C_ERR),(pms,'PMS',C_GOLD)]:
        fig.add_trace(go.Scatter(
            x=[pt['riesgo']],y=[pt['rendimiento']],mode='markers+text',
            marker=dict(color=col,size=18,symbol='star',line=dict(width=2,color='white')),
            text=[lbl],textposition='top right',textfont=dict(color=col,size=13),
            name=f'{lbl} Sharpe={pt["sharpe"]:.2f}'))
    for i,tk in enumerate(tickers):
        fig.add_trace(go.Scatter(
            x=[stats.loc[tk,'Std_Anual']],y=[stats.loc[tk,'Rend_Anual']],
            mode='markers+text',
            marker=dict(color=COLORES[i%len(COLORES)],size=10,symbol='diamond',
                        line=dict(width=1,color='white')),
            text=[tk],textposition='top center',textfont=dict(size=10),showlegend=False))
    fig.update_xaxes(title='Riesgo sigma anualizado',tickformat='.0%')
    fig.update_yaxes(title='Rendimiento anual',tickformat='.0%')
    return apl(fig,'Frontera Eficiente de Markowitz + CML',alto=600)


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO O — PDF INSTITUCIONAL
# ══════════════════════════════════════════════════════════════════════════════

class PortfolioPDF(FPDF):
    C_BG=(13,17,23); C_PANEL=(22,27,34); C_ACCENT=(88,166,255)
    C_GOLD=(255,215,0); C_TEXT=(201,209,217); C_MUTED=(139,148,158)

    def header(self):
        self.set_fill_color(*self.C_BG); self.rect(0,0,210,297,'F')
        self.set_fill_color(*self.C_PANEL); self.rect(0,0,210,16,'F')
        self.set_fill_color(*self.C_ACCENT); self.rect(0,16,210,0.8,'F')
        self.set_font('Helvetica','B',8); self.set_text_color(*self.C_ACCENT)
        self.set_y(4); self.cell(0,6,_t('PORTFOLIO ANALYSIS v3 -- INSTITUTIONAL'),align='L')
        self.set_text_color(*self.C_MUTED); self.set_font('Helvetica','',7)
        self.set_y(4); self.cell(0,6,_t(datetime.today().strftime('%d/%m/%Y')),align='R')
        self.ln(14)

    def footer(self):
        self.set_fill_color(*self.C_PANEL); self.rect(0,285,210,12,'F')
        self.set_fill_color(*self.C_ACCENT); self.rect(0,285,210,0.5,'F')
        self.set_y(-10); self.set_font('Helvetica','',7)
        self.set_text_color(*self.C_MUTED)
        self.cell(0,5,_t(f'Modulo 6 -- Finanzas Cuantitativas | Pag {self.page_no()}'),align='C')

    def sec(self, txt):
        self.set_fill_color(*self.C_ACCENT); self.rect(self.get_x(),self.get_y(),3,7,'F')
        self.set_x(self.get_x()+6); self.set_font('Helvetica','B',13)
        self.set_text_color(*self.C_ACCENT); self.cell(0,7,_t(txt),ln=True); self.ln(4)

    def tabla(self, df, dec=4):
        cols=[str(df.index.name or 'Index')]+[str(c) for c in df.columns]
        aw=190/len(cols)
        self.set_fill_color(*self.C_PANEL); self.set_font('Helvetica','B',8)
        self.set_text_color(*self.C_ACCENT)
        for c in cols: self.cell(aw,7,_t(c[:16]),border=1,align='C',fill=True)
        self.ln()
        self.set_font('Helvetica','',7)
        for i,(idx,row) in enumerate(df.iterrows()):
            self.set_fill_color(*(self.C_BG if i%2==0 else self.C_PANEL))
            self.set_text_color(*self.C_TEXT)
            self.cell(aw,6,_t(str(idx)[:16]),border=1,align='L',fill=True)
            for val in row:
                try: txt2=f'{float(val):.{dec}f}'
                except: txt2=_t(str(val)[:14])
                self.cell(aw,6,txt2,border=1,align='R',fill=True)
            self.ln()
        self.ln(5)

    def kpi_row(self,items):
        n=len(items); aw=(190-4*(n-1))/n; y0,x0=self.get_y(),self.get_x()
        for i,(val,lbl,sub) in enumerate(items):
            xi=x0+i*(aw+4); self.set_xy(xi,y0)
            self.set_fill_color(*self.C_PANEL); self.rect(xi,y0,aw,22,'F')
            self.set_font('Helvetica','',6); self.set_text_color(*self.C_MUTED)
            self.set_xy(xi,y0+1); self.cell(aw,6,_t(lbl.upper()),align='C')
            self.set_font('Helvetica','B',13); self.set_text_color(*self.C_ACCENT)
            self.set_xy(xi,y0+7); self.cell(aw,8,_t(val),align='C')
            self.set_font('Helvetica','',6); self.set_text_color(*self.C_MUTED)
            self.set_xy(xi,y0+16); self.cell(aw,5,_t(sub),align='C')
        self.set_y(y0+28)

    def img_safe(self,path,**kw):
        if os.path.exists(path): self.image(path,**kw)


# ══════════════════════════════════════════════════════════════════════════════
# CALLBACK PRINCIPAL
# ══════════════════════════════════════════════════════════════════════════════

def ejecutar_analisis(b):
    with out_area:
        clear_output(wait=True)

        # ── Leer widgets ──────────────────────────────────────────────────────
        TICKERS    = list(w_tickers.value)
        PERIODO    = w_periodo.value
        TASA_RF    = w_rf.value / 100
        PESO_MIN   = w_peso_min.value / 100
        PESO_MAX   = w_peso_max.value / 100
        BL_TAU     = w_bl_tau.value
        BL_VIEWS   = w_bl_views.value
        BL_CONF    = w_bl_conf.value
        N_SIMS     = w_mc_sims.value
        AÑOS_MC    = sorted(list(w_mc_años.value))
        NAV        = w_nav.value
        COMISION   = w_comision.value

        if len(TICKERS)<2:
            w_status.value='<span style="color:#ff7b72">❌ Min 2 tickers</span>'; return

        FACTOR   = {'diaria':252,'semanal':52,'mensual':12}[PERIODO]
        RF_P     = (1+TASA_RF)**(1/FACTOR)-1
        RF_A     = (1+RF_P)**FACTOR-1
        IVL      = {'diaria':'1d','semanal':'1wk','mensual':'1mo'}[PERIODO]
        FECHA_FIN= datetime.today().strftime('%Y-%m-%d')
        FECHA_INI= (datetime.today()-timedelta(days=365*5)).strftime('%Y-%m-%d')

        w_status.value='<span style="color:#ffa657">⏳ Ejecutando análisis institucional...</span>'
        print(f'\n{'='*60}'); print(f'  ANÁLISIS INSTITUCIONAL v3')
        print(f'  {len(TICKERS)} activos | {PERIODO} | RF={TASA_RF:.2%} | NAV=${NAV:,.0f}')
        print(f'{'='*60}')

        # ── A. DATOS ──────────────────────────────────────────────────────────
        print('\n[A] Descargando datos...')
        precios = descargar_precios(TICKERS,FECHA_INI,FECHA_FIN,IVL)
        TICKERS = precios.columns.tolist(); N=len(TICKERS)
        rend    = np.log(precios/precios.shift(1)).dropna()
        stats   = estadisticos_base(rend,FACTOR,RF_P)
        print(stats[['Rend_Anual','Std_Anual','Sharpe','Skewness','Kurtosis']].to_string())

        # Datos diarios siempre (para stress/drawdown)
        if PERIODO != 'diaria':
            print('  Descargando precios diarios adicionales para stress/drawdown...')
            precios_d = descargar_precios(TICKERS,FECHA_INI,FECHA_FIN,'1d')
            rend_d    = np.log(precios_d/precios_d.shift(1)).dropna()
        else:
            precios_d = precios; rend_d = rend

        # ── B. COVARIANZA LEDOIT-WOLF ──────────────────────────────────────────
        print('\n[B] Ledoit-Wolf shrinkage...')
        corr    = rend.corr()
        cov_muestral = rend.cov()*FACTOR
        cov_lw, shrinkage = cov_ledoit_wolf(rend,FACTOR)
        print(f'  Coeficiente de shrinkage: {shrinkage:.4f}')
        print(f'  (0=muestral puro, 1=identidad escalada — optimo ~0.05-0.30)')
        Sigma = cov_lw.values
        mu    = stats['Rend_Anual'].values

        fig_corr_plot = fig_heatmap_corr(corr)
        fig_corr_plot.show()

        # ── C. OPTIMIZACIÓN ───────────────────────────────────────────────────
        print('\n[C] Optimizando portafolios...')
        pmv = optimizar(mu,Sigma,RF_A,N,PESO_MIN,PESO_MAX,'varianza')
        pms = optimizar(mu,Sigma,RF_A,N,PESO_MIN,PESO_MAX,'sharpe')
        prp = optimizar(mu,Sigma,RF_A,N,PESO_MIN,PESO_MAX,'risk_parity')

        # Black-Litterman
        print('  Black-Litterman...')
        # Retornos de equilibrio: lambda * Sigma @ w_mkt (proxy: pesos iguales)
        w_eq    = np.full(N,1/N)
        lam_    = (RF_A + 0.05) / float(w_eq@Sigma@w_eq)  # lambda implícita
        mu_eq   = lam_ * Sigma @ w_eq
        mu_bl   = black_litterman(mu_eq,Sigma,RF_A,TICKERS,BL_VIEWS,BL_TAU,BL_CONF)
        pbl     = optimizar(mu_bl,Sigma,RF_A,N,PESO_MIN,PESO_MAX,'sharpe')
        pbl['mu_bl'] = mu_bl

        print('\n  Resultados:')
        for p,lbl in [(pmv,'PMV'),(pms,'PMS'),(prp,'Risk Parity'),(pbl,'Black-Litterman')]:
            print(f'  {lbl:20s} Rend={p["rendimiento"]:.2%}  sigma={p["riesgo"]:.2%}  Sharpe={p["sharpe"]:.3f}')

        # Gráfica pesos comparativa
        fig_pw = go.Figure()
        for p,lbl,col in [(pmv,'PMV',C_ERR),(pms,'PMS',C_GOLD),
                          (prp,'Risk Parity',C_OK),(pbl,'Black-Litterman',C_BLUE)]:
            fig_pw.add_trace(go.Bar(name=lbl,x=TICKERS,y=p['pesos'],
                                    text=[f'{w:.1%}' for w in p['pesos']],
                                    textposition='outside',textfont=dict(size=9)))
        fig_pw.update_yaxes(title='Peso',tickformat='.0%')
        apl(fig_pw,'Comparacion de Pesos — 4 Portafolios',alto=460,barmode='group').show()

        # ── D. FRONTERA ───────────────────────────────────────────────────────
        print('\n[D] Frontera eficiente...')
        df_sim,_ = simular_frontera(mu,Sigma,RF_A,N,PESO_MIN,PESO_MAX,500)
        fig_fron_plot = fig_frontera(df_sim,pmv,pms,stats,TICKERS,RF_A)
        fig_fron_plot.show()

        # ── E. RIESGO AVANZADO ────────────────────────────────────────────────
        print('\n[E] CVaR / VaR avanzado...')
        rend_pmv_d = (rend_d * pmv['pesos'][:len(rend_d.columns)]).sum(axis=1)
        rend_pms_d = (rend_d * pms['pesos'][:len(rend_d.columns)]).sum(axis=1)

        riesgo_pmv = calcular_riesgo_completo(rend_pmv_d,pmv['rendimiento'],pmv['riesgo'],RF_A,'PMV')
        riesgo_pms = calcular_riesgo_completo(rend_pms_d,pms['rendimiento'],pms['riesgo'],RF_A,'PMS')
        df_riesgo  = pd.DataFrame([riesgo_pmv,riesgo_pms]).set_index('Portafolio')

        print(df_riesgo.to_string())
        fig_var_distribucion(rend_pms_d,'PMS').show()
        fig_var_distribucion(rend_pmv_d,'PMV').show()

        # ── F. STRESS TESTING ─────────────────────────────────────────────────
        print('\n[F] Stress testing...')
        # Alinear pesos con tickers disponibles en precios_d
        tks_d  = [t for t in TICKERS if t in precios_d.columns]
        w_pmv_d= np.array([pmv['pesos'][TICKERS.index(t)] for t in tks_d])
        w_pms_d= np.array([pms['pesos'][TICKERS.index(t)] for t in tks_d])
        w_pmv_d/=w_pmv_d.sum(); w_pms_d/=w_pms_d.sum()

        df_stress = stress_testing(precios_d[tks_d],w_pmv_d,w_pms_d,tks_d)
        print(df_stress.to_string())
        fig_stress(df_stress).show()

        # ── G. DRAWDOWN ───────────────────────────────────────────────────────
        print('\n[G] Drawdown analysis...')
        dd_pmv,max_dd_pmv,calmar_pmv,rec_pmv = calcular_drawdown(rend_pmv_d)
        dd_pms,max_dd_pms,calmar_pms,rec_pms = calcular_drawdown(rend_pms_d)

        print(f'  PMV — Max DD: {max_dd_pmv:.2%}  Calmar: {calmar_pmv:.2f}  Rec: {rec_pmv} dias')
        print(f'  PMS — Max DD: {max_dd_pms:.2%}  Calmar: {calmar_pms:.2f}  Rec: {rec_pms} dias')
        fig_drawdown(rend_pmv_d,rend_pms_d,rend_d,TICKERS).show()

        # ── H. ROLLING ANALYSIS ───────────────────────────────────────────────
        print('\n[H] Rolling analysis...')
        roll_pmv = rolling_metrics(rend_pmv_d,252,RF_A)
        roll_pms = rolling_metrics(rend_pms_d,252,RF_A)
        fig_rolling(roll_pmv,roll_pms,'sharpe').show()
        fig_rolling(roll_pmv,roll_pms,'vol').show()

        # ── I. FACTORES FAMA-FRENCH ───────────────────────────────────────────
        print('\n[I] Factores Fama-French proxy...')
        factores = descargar_factores_ff(FECHA_INI,FECHA_FIN)
        alpha_pmv=alpha_pms=None; betas_pmv_df=betas_pms_df=None
        r2_pmv=r2_pms=None
        if factores is not None:
            betas_pmv_df,r2_pmv,alpha_pmv = regresion_factores(rend_pmv_d,factores)
            betas_pms_df,r2_pms,alpha_pms = regresion_factores(rend_pms_d,factores)
            print(f'  PMV — R2={r2_pmv:.3f}  Alpha anual={alpha_pmv:.2%}')
            print(f'  PMS — R2={r2_pms:.3f}  Alpha anual={alpha_pms:.2%}')
            print('\n  PMV Betas:')
            print(betas_pmv_df.to_string())
            fig_factor_betas(betas_pmv_df,betas_pms_df).show()
        else:
            print('  No se pudieron descargar factores FF')

        # ── J. RÉGIMEN DE MERCADO ─────────────────────────────────────────────
        print('\n[J] Regimenes de mercado...')
        rend_pms_port = pd.Series((rend_d.reindex(columns=tks_d)*w_pms_d).sum(axis=1),name='PMS')
        reg_df,reg_stats = detectar_regimenes(rend_pms_port)
        print(reg_stats.to_string())
        cum_port = np.exp(rend_pms_port.cumsum())*100
        reg_df_aligned = reg_df.reindex(rend_pms_port.index).ffill()
        fig_regimenes(cum_port,reg_df_aligned).show()

        # ── K. MONTE CARLO FORWARD ────────────────────────────────────────────
        print(f'\n[K] Monte Carlo ({N_SIMS:,} sims | horizontes: {AÑOS_MC})...')
        mc_pmv = monte_carlo_forward(pmv['rendimiento'],pmv['riesgo'],NAV,N_SIMS,AÑOS_MC,'PMV')
        mc_pms = monte_carlo_forward(pms['rendimiento'],pms['riesgo'],NAV,N_SIMS,AÑOS_MC,'PMS')

        for años in AÑOS_MC:
            for mc,lbl in [(mc_pmv,'PMV'),(mc_pms,'PMS')]:
                r=mc[años]
                print(f'  {lbl} {años}Y — Med: ${r["p50"]:,.0f}  P5: ${r["p5"]:,.0f}  '
                      f'P95: ${r["p95"]:,.0f}  Prob>0: {r["prob_positivo"]:.1%}  Prob 2x: {r["prob_2x"]:.1%}')

        fig_monte_carlo(mc_pms,NAV,AÑOS_MC,'PMS').show()

        # ── L. TRADE LIST REBALANCEO ──────────────────────────────────────────
        print('\n[L] Trade list de rebalanceo...')
        # Precio actual = última fila de precios
        precios_actuales = precios_d.iloc[-1].to_dict()
        df_reb_pms = calcular_rebalanceo(TICKERS,pms['pesos'],precios_actuales,
                                          NAV,COMISION)
        df_reb_pmv = calcular_rebalanceo(TICKERS,pmv['pesos'],precios_actuales,
                                          NAV,COMISION)
        print('\n  Trade List hacia PMS:')
        print(df_reb_pms[['Peso Actual','Peso Objetivo','Gap','Dirección',
                           'USD Trade','Comisión Est.']].to_string())
        print(f'  Turnover total: ${df_reb_pms["USD Trade"].sum():,.0f}  '
              f'({df_reb_pms["USD Trade"].sum()/NAV:.1%} del NAV)')
        print(f'  Costo estimado: ${df_reb_pms["Comisión Est."].sum():,.0f}')
        fig_rebalanceo(df_reb_pms,'Rebalanceo hacia PMS (Max Sharpe)').show()

        # ── M. ANÁLISIS TÉCNICO ───────────────────────────────────────────────
        print('\n[M] Analisis tecnico...')
        inds = {}
        for tk in tqdm(TICKERS,desc='Indicadores'):
            if tk not in precios_d.columns: continue
            c = precios_d[tk].dropna()
            v = pd.Series(dtype=float)
            try:
                raw=yf.download(tk,start=FECHA_INI,end=FECHA_FIN,interval='1d',
                                progress=False,auto_adjust=False)
                if 'Volume' in raw.columns: v=raw['Volume'].squeeze()
            except: pass
            rsi_s=calcular_rsi(c); ml,ms,mh=calcular_macd(c); pk,pd_=calcular_estocastico(c)
            inds[tk]={'close':c,'volume':v,'sma50':c.rolling(50).mean(),
                      'sma100':c.rolling(100).mean(),'sma200':c.rolling(200).mean(),
                      'rsi':rsi_s,'macd':ml,'macd_s':ms,'macd_h':mh,'pct_k':pk,'pct_d':pd_,
                      'señales':detectar_señales(c,rsi_s,ml,ms)}

        fig_tecnica(TICKERS[0],inds[TICKERS[0]]).show()
        fig_base100(inds,TICKERS).show()

        # ── N. DASHBOARD HTML ─────────────────────────────────────────────────
        print('\n[N] Generando dashboard HTML...')
        to_div = lambda fig: pio.to_html(fig,full_html=False,include_plotlyjs=False)

        # Pre-render de todas las figuras para el HTML
        print('  Renderizando figuras...')
        divs = {
            'fron':   to_div(fig_fron_plot),
            'corr':   to_div(fig_corr_plot),
            'b100':   to_div(fig_base100(inds,TICKERS)),
            'pesos':  to_div(fig_pw),
            'var_pms':to_div(fig_var_distribucion(rend_pms_d,'PMS')),
            'var_pmv':to_div(fig_var_distribucion(rend_pmv_d,'PMV')),
            'stress': to_div(fig_stress(df_stress)),
            'dd':     to_div(fig_drawdown(rend_pmv_d,rend_pms_d,rend_d,TICKERS)),
            'roll_sh':to_div(fig_rolling(roll_pmv,roll_pms,'sharpe')),
            'roll_vo':to_div(fig_rolling(roll_pmv,roll_pms,'vol')),
            'mc_pms': to_div(fig_monte_carlo(mc_pms,NAV,AÑOS_MC,'PMS')),
            'mc_pmv': to_div(fig_monte_carlo(mc_pmv,NAV,AÑOS_MC,'PMV')),
            'reb':    to_div(fig_rebalanceo(df_reb_pms,'Rebalanceo hacia PMS')),
        }
        if betas_pmv_df is not None:
            divs['ff'] = to_div(fig_factor_betas(betas_pmv_df,betas_pms_df))
        divs['reg'] = to_div(fig_regimenes(cum_port,reg_df_aligned))
        for tk in tqdm(TICKERS,desc='Tech HTML'):
            if tk in inds: divs[f'tec_{tk}']=to_div(fig_tecnica(tk,inds[tk]))

        # Tablas HTML
        def df2html(df,fmt=None):
            d=df.copy()
            if fmt:
                for c,f in fmt.items():
                    if c in d.columns: d[c]=d[c].map(f)
            return d.to_html(classes='dt',border=0)

        pct = '{:.2%}'.format; flt = '{:.4f}'.format
        t_stats  = df2html(stats,{'Rend_Anual':pct,'Std_Anual':pct,'Sharpe':flt,
                                  'Skewness':flt,'Kurtosis':flt})
        t_riesgo = df2html(df_riesgo.applymap(lambda x: f'{x:.2%}' if isinstance(x,float) else x))
        t_stress = df2html(df_stress,{'Retorno Total':pct,'Max Drawdown':pct,'Vol Anual':pct})
        t_reb    = df2html(df_reb_pms[['Peso Actual','Peso Objetivo','Gap','Dirección',
                                        'USD Trade','Comisión Est.']],
                            {'Peso Actual':pct,'Peso Objetivo':pct,'Gap':pct})

        def tp(p,nm):
            rows=''.join(f'<tr><td>{tk}</td><td>{w:.2%}</td>'
                        f'<td><div class="br" style="width:{w*260:.0f}px"></div></td></tr>'
                        for tk,w in zip(TICKERS,p['pesos']))
            return (f'<div class="card"><h3>{nm}</h3>'
                   f'<p>Rend <b>{p["rendimiento"]:.2%}</b> | '
                   f'sigma <b>{p["riesgo"]:.2%}</b> | '
                   f'Sharpe <b>{p["sharpe"]:.3f}</b></p>'
                   f'<table class="dt"><tr><th>Ticker</th><th>Peso</th><th></th></tr>{rows}</table></div>')

        # MC summary cards
        mc_cards = ''
        for años in AÑOS_MC:
            r=mc_pms[años]
            mc_cards += (f'<div class="kpi"><div class="lb">PMS {años}Y Mediana</div>'
                        f'<div class="vl">${r["p50"]:,.0f}</div>'
                        f'<div class="su">P5: ${r["p5"]:,.0f} | P95: ${r["p95"]:,.0f}</div></div>')

        # Drawdown cards
        dd_html = (f'<div class="kpis" style="margin:0 0 20px">' 
                  f'<div class="kpi"><div class="lb">PMV Max Drawdown</div><div class="vl" style="color:#ff7b72">{max_dd_pmv:.2%}</div><div class="su">Calmar: {calmar_pmv:.2f}</div></div>'
                  f'<div class="kpi"><div class="lb">PMS Max Drawdown</div><div class="vl" style="color:#ff7b72">{max_dd_pms:.2%}</div><div class="su">Calmar: {calmar_pms:.2f}</div></div>'
                  f'<div class="kpi"><div class="lb">PMV Recuperación</div><div class="vl">{rec_pmv or "No rec."}</div><div class="su">Días desde trough</div></div>'
                  f'<div class="kpi"><div class="lb">PMS Recuperación</div><div class="vl">{rec_pms or "No rec."}</div><div class="su">Días desde trough</div></div>'
                  f'</div>')

        # FF alpha cards
        ff_html_extra = ''
        if alpha_pmv is not None:
            ff_html_extra = (f'<div class="kpis" style="margin:0 0 20px">'
                           f'<div class="kpi"><div class="lb">PMV Alpha anual</div><div class="vl">{alpha_pmv:.2%}</div><div class="su">R2={r2_pmv:.3f}</div></div>'
                           f'<div class="kpi"><div class="lb">PMS Alpha anual</div><div class="vl">{alpha_pms:.2%}</div><div class="su">R2={r2_pms:.3f}</div></div>'
                           f'</div>')

        # Señales técnicas
        señales_html=''.join(f'<div class="sbox"><h4>{tk}</h4>'+''.join(f'<p>{s}</p>' for s in inds[tk]['señales'])+'</div>'
                             for tk in TICKERS if tk in inds)
        opts_tk=''.join(f'<option value="g_{tk}">{tk}</option>' for tk in TICKERS)
        divs_tk=''.join(f'<div id="g_{tk}" class="gtk" style="display:none">{divs.get(f"tec_{tk}","")}</div>' for tk in TICKERS)

        tickers_str=' | '.join(TICKERS); fecha_rep=datetime.today().strftime('%d/%m/%Y')

        html = f"""<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <title>Portfolio Institucional v3</title>
  <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
  <style>
    :root{{--bg:#0d1117;--panel:#161b22;--b:#30363d;--acc:#58a6ff;--gold:#ffd700;
           --txt:#c9d1d9;--muted:#8b949e;--ok:#3fb950;--err:#ff7b72;}}
    *{{box-sizing:border-box;margin:0;padding:0;}}
    body{{font-family:'Inter',system-ui,sans-serif;background:var(--bg);color:var(--txt);}}
    header{{background:var(--panel);border-bottom:1px solid var(--b);padding:20px 36px;}}
    header h1{{font-size:1.5rem;color:var(--acc);}}
    header p{{color:var(--muted);font-size:.82rem;margin-top:4px;}}
    .kpis{{display:flex;gap:10px;flex-wrap:wrap;padding:16px 36px;
           background:var(--panel);border-bottom:1px solid var(--b);}}
    .kpi{{background:var(--bg);border:1px solid var(--b);border-radius:8px;
          padding:10px 16px;min-width:130px;}}
    .kpi .lb{{font-size:.68rem;color:var(--muted);text-transform:uppercase;letter-spacing:.7px;}}
    .kpi .vl{{font-size:1.25rem;font-weight:700;color:var(--acc);margin-top:2px;}}
    .kpi .su{{font-size:.68rem;color:var(--muted);}}
    .tabs{{display:flex;padding:0 36px;border-bottom:1px solid var(--b);
           background:var(--panel);overflow-x:auto;}}
    .tab{{padding:12px 20px;cursor:pointer;font-size:.85rem;color:var(--muted);
          border-bottom:2px solid transparent;transition:all .2s;white-space:nowrap;}}
    .tab:hover{{color:var(--txt);}} .tab.active{{color:var(--acc);border-bottom-color:var(--acc);}}
    .content{{padding:26px 36px;}}
    .panel{{display:none;animation:fi .3s ease;}} .panel.active{{display:block;}}
    @keyframes fi{{from{{opacity:0;transform:translateY(5px)}}to{{opacity:1;transform:none}}}}
    .grid2{{display:grid;grid-template-columns:1fr 1fr;gap:18px;margin:14px 0;}}
    .card{{background:var(--panel);border:1px solid var(--b);border-radius:8px;padding:18px;}}
    .card h3{{color:var(--gold);margin-bottom:8px;font-size:.95rem;}}
    .card p{{font-size:.82rem;color:var(--muted);margin-bottom:8px;}} .card p b{{color:var(--txt);}}
    .st{{font-size:1.05rem;color:var(--acc);border-left:3px solid var(--acc);
         padding-left:10px;margin:16px 0 12px;}}
    .dt{{width:100%;border-collapse:collapse;font-size:.8rem;margin-top:10px;}}
    .dt th{{background:#1c2333;color:var(--acc);padding:8px 12px;text-align:left;
            border:1px solid var(--b);font-weight:600;}}
    .dt td{{padding:7px 12px;border:1px solid var(--b);}}
    .dt tr:nth-child(even) td{{background:rgba(255,255,255,.02);}}
    .dt tr:hover td{{background:rgba(88,166,255,.05);}}
    .br{{height:6px;background:linear-gradient(90deg,var(--acc),#1f6feb);border-radius:3px;min-width:2px;}}
    .sbox{{background:var(--panel);border:1px solid var(--b);border-radius:7px;padding:12px 16px;margin-bottom:10px;}}
    .sbox h4{{color:var(--gold);margin-bottom:7px;font-size:.9rem;}} .sbox p{{font-size:.8rem;margin-bottom:2px;}}
    select{{background:var(--panel);color:var(--txt);border:1px solid var(--b);
            border-radius:6px;padding:6px 12px;font-size:.85rem;margin-bottom:14px;cursor:pointer;}}
    footer{{padding:16px 36px;border-top:1px solid var(--b);color:var(--muted);font-size:.72rem;text-align:center;}}
    .badge-ok{{color:var(--ok);}} .badge-err{{color:var(--err);}} .badge-warn{{color:var(--gold);}}
  </style>
</head>
<body>
<header>
  <h1>Portfolio Dashboard v3 — Institucional</h1>
  <p>{tickers_str} | {PERIODO} | RF:{TASA_RF:.2%} | NAV:${NAV:,.0f} | Shrinkage LW:{shrinkage:.3f} | {fecha_rep}</p>
</header>
<div class="kpis">
  <div class="kpi"><div class="lb">Activos</div><div class="vl">{N}</div><div class="su">{PERIODO}</div></div>
  <div class="kpi"><div class="lb">PMS Sharpe</div><div class="vl">{pms['sharpe']:.3f}</div><div class="su">Max Sharpe</div></div>
  <div class="kpi"><div class="lb">PMS Rend.</div><div class="vl">{pms['rendimiento']:.1%}</div><div class="su">Anual</div></div>
  <div class="kpi"><div class="lb">PMS CVaR 99%</div><div class="vl" style="color:var(--err)">{riesgo_pms['CVaR 99% (ES)']:.2%}</div><div class="su">Expected Shortfall</div></div>
  <div class="kpi"><div class="lb">PMS Max DD</div><div class="vl" style="color:var(--err)">{max_dd_pms:.2%}</div><div class="su">Calmar:{calmar_pms:.2f}</div></div>
  <div class="kpi"><div class="lb">BL Rend.</div><div class="vl">{pbl['rendimiento']:.1%}</div><div class="su">Black-Litterman</div></div>
  <div class="kpi"><div class="lb">RF Anual</div><div class="vl">{TASA_RF:.2%}</div><div class="su">Risk-free</div></div>
  {mc_cards}
</div>
<div class="tabs">
  <div class="tab active" onclick="tab('stats',this)">📐 Stats</div>
  <div class="tab" onclick="tab('corr',this)">🔗 Correlacion</div>
  <div class="tab" onclick="tab('opt',this)">⚙️ Optimizacion</div>
  <div class="tab" onclick="tab('fron',this)">🌐 Frontera</div>
  <div class="tab" onclick="tab('riesgo',this)">📉 Riesgo</div>
  <div class="tab" onclick="tab('stress',this)">🔥 Stress</div>
  <div class="tab" onclick="tab('dd',this)">🌊 Drawdown</div>
  <div class="tab" onclick="tab('rolling',this)">📊 Rolling</div>
  <div class="tab" onclick="tab('ff',this)">🔬 Factores</div>
  <div class="tab" onclick="tab('reg',this)">🎯 Regimen</div>
  <div class="tab" onclick="tab('mc',this)">🎲 Monte Carlo</div>
  <div class="tab" onclick="tab('reb',this)">⚖️ Rebalanceo</div>
  <div class="tab" onclick="tab('tec',this)">📈 Tecnico</div>
</div>
<div class="content">
  <div id="tab-stats"   class="panel active"><div class="st">Estadisticos por Activo</div>{t_stats}<br>{divs['b100']}</div>
  <div id="tab-corr"    class="panel"><div class="st">Correlacion + Ledoit-Wolf (shrinkage={shrinkage:.3f})</div>{divs['corr']}</div>
  <div id="tab-opt"     class="panel"><div class="st">4 Portafolios Optimizados</div>{divs['pesos']}
    <div class="grid2">{tp(pmv,'Min Varianza PMV')}{tp(pms,'Max Sharpe PMS')}{tp(prp,'Risk Parity')}{tp(pbl,'Black-Litterman')}</div></div>
  <div id="tab-fron"    class="panel"><div class="st">Frontera Eficiente + CML</div>{divs['fron']}</div>
  <div id="tab-riesgo"  class="panel"><div class="st">Riesgo Avanzado — VaR / CVaR / ES</div>{t_riesgo}<br>{divs['var_pms']}<br>{divs['var_pmv']}</div>
  <div id="tab-stress"  class="panel"><div class="st">Stress Testing Historico</div>{divs['stress']}<br>{t_stress}</div>
  <div id="tab-dd"      class="panel"><div class="st">Drawdown Analysis</div>{dd_html}{divs['dd']}</div>
  <div id="tab-rolling" class="panel"><div class="st">Rolling Sharpe</div>{divs['roll_sh']}<div class="st">Rolling Volatilidad</div>{divs['roll_vo']}</div>
  <div id="tab-ff"      class="panel"><div class="st">Factores Fama-French (Proxy ETF)</div>{ff_html_extra}{divs.get('ff','<p>No disponible</p>')}</div>
  <div id="tab-reg"     class="panel"><div class="st">Regimenes de Mercado</div>{divs['reg']}</div>
  <div id="tab-mc"      class="panel"><div class="st">Monte Carlo GBM — PMS</div>{divs['mc_pms']}<div class="st">Monte Carlo GBM — PMV</div>{divs['mc_pmv']}</div>
  <div id="tab-reb"     class="panel"><div class="st">Trade List — Rebalanceo hacia PMS</div>{t_reb}<br>{divs['reb']}</div>
  <div id="tab-tec"     class="panel"><div class="st">Analisis Tecnico</div>
    <select onchange="showChart(this.value)">{opts_tk}</select>{divs_tk}
    <div class="st">Senales (90 dias)</div>{señales_html}</div>
</div>
<footer>Portfolio Institucional v3 | Markowitz + BL + CVaR + Stress + FF + MC | {fecha_rep}</footer>
<script>
  document.querySelector('.gtk')&&(document.querySelector('.gtk').style.display='block');
  function tab(id,el){{
    document.querySelectorAll('.panel').forEach(p=>p.classList.remove('active'));
    document.querySelectorAll('.tab').forEach(t=>t.classList.remove('active'));
    document.getElementById('tab-'+id).classList.add('active');el.classList.add('active');
  }}
  function showChart(id){{
    document.querySelectorAll('.gtk').forEach(g=>g.style.display='none');
    const t=document.getElementById(id);if(t)t.style.display='block';
  }}
</script>
</body></html>"""

        HTML_PATH='portfolio_dashboard_v3.html'
        with open(HTML_PATH,'w',encoding='utf-8') as f: f.write(html)
        print(f'✅ HTML: {HTML_PATH} ({os.path.getsize(HTML_PATH)//1024} KB)')

        # ── O. EXCEL ──────────────────────────────────────────────────────────
        print('\n[O] Excel...')
        EXCEL_PATH='portfolio_v3.xlsx'
        with pd.ExcelWriter(EXCEL_PATH,engine='openpyxl') as writer:
            precios.round(4).to_excel(writer,sheet_name='Precios')
            rend.round(6).to_excel(writer,sheet_name='Rendimientos')
            stats.round(6).to_excel(writer,sheet_name='Estadisticas')
            corr.round(4).to_excel(writer,sheet_name='Correlacion')
            cov_lw.round(8).to_excel(writer,sheet_name='Cov_LedoitWolf')
            df_sim.round(6).to_excel(writer,sheet_name='Frontera',index=False)
            df_riesgo.round(6).to_excel(writer,sheet_name='VaR_CVaR')
            df_stress.round(4).to_excel(writer,sheet_name='Stress_Testing')
            df_reb_pms.round(4).to_excel(writer,sheet_name='Rebalanceo_PMS')
            df_reb_pmv.round(4).to_excel(writer,sheet_name='Rebalanceo_PMV')
            reg_stats.round(4).to_excel(writer,sheet_name='Regimenes')
            for p,nm in [(pmv,'PMV'),(pms,'PMS'),(prp,'RiskParity'),(pbl,'BlackLitterman')]:
                dp=pd.DataFrame({'Ticker':TICKERS,'Peso':p['pesos']})
                dp.to_excel(writer,sheet_name=nm,index=False)
            # MC summary
            mc_rows=[]
            for años in AÑOS_MC:
                for mc,lbl in [(mc_pmv,'PMV'),(mc_pms,'PMS')]:
                    r=mc[años]
                    mc_rows.append({'Port':lbl,'Anos':años,'P5':r['p5'],'P25':r['p25'],
                                    'P50':r['p50'],'P75':r['p75'],'P95':r['p95'],
                                    'Prob_positivo':r['prob_positivo'],'Prob_2x':r['prob_2x']})
            pd.DataFrame(mc_rows).to_excel(writer,sheet_name='MonteCarlo',index=False)
            if betas_pmv_df is not None:
                betas_pmv_df.round(4).to_excel(writer,sheet_name='FF_PMV')
                betas_pms_df.round(4).to_excel(writer,sheet_name='FF_PMS')

        # Estilo Excel
        wb=openpyxl.load_workbook(EXCEL_PATH)
        for ws in wb.worksheets:
            fh=PatternFill('solid',fgColor='1c2333')
            fa=PatternFill('solid',fgColor='0d1117')
            fn=PatternFill('solid',fgColor='161b22')
            for ri,row in enumerate(ws.iter_rows()):
                for cell in row:
                    cell.border=Border(left=Side(style='thin',color='30363d'),
                                      right=Side(style='thin',color='30363d'),
                                      top=Side(style='thin',color='30363d'),
                                      bottom=Side(style='thin',color='30363d'))
                    if ri==0:
                        cell.fill=fh; cell.font=Font(bold=True,color='58a6ff',name='Calibri',size=10)
                    else:
                        cell.fill=fa if ri%2==0 else fn
                        cell.font=Font(color='c9d1d9',name='Calibri',size=9)
                    cell.alignment=Alignment(horizontal='right',vertical='center')
            for col in ws.columns:
                mx=max((len(str(c.value or '')) for c in col),default=8)
                ws.column_dimensions[get_column_letter(col[0].column)].width=min(mx+3,28)
        wb.save(EXCEL_PATH)
        print(f'✅ Excel: {EXCEL_PATH} ({os.path.getsize(EXCEL_PATH)//1024} KB)')

        # ── P. PDF ────────────────────────────────────────────────────────────
        print('\n[P] PDF institucional...')
        IMG_DIR='pdf_imgs_v3'; os.makedirs(IMG_DIR,exist_ok=True)
        IMGS_OK=False
        try:
            import kaleido  # noqa
            fig_fron_plot.write_image(f'{IMG_DIR}/frontera.png',width=900,height=480,scale=2)
            fig_corr_plot.write_image(f'{IMG_DIR}/corr.png',width=900,height=max(400,52*N),scale=2)
            fig_stress(df_stress).write_image(f'{IMG_DIR}/stress.png',width=900,height=380,scale=2)
            fig_drawdown(rend_pmv_d,rend_pms_d).write_image(f'{IMG_DIR}/dd.png',width=900,height=430,scale=2)
            fig_rolling(roll_pmv,roll_pms,'sharpe').write_image(f'{IMG_DIR}/rolling.png',width=900,height=550,scale=2)
            if betas_pmv_df is not None:
                fig_factor_betas(betas_pmv_df,betas_pms_df).write_image(f'{IMG_DIR}/ff.png',width=900,height=400,scale=2)
            fig_monte_carlo(mc_pms,NAV,AÑOS_MC,'PMS').write_image(f'{IMG_DIR}/mc.png',width=1200,height=380*len(AÑOS_MC),scale=1.5)
            IMGS_OK=True; print('  Imagenes PNG OK')
        except Exception as e:
            print(f'  kaleido: {e} -- PDF sin graficas embebidas')

        pdf=PortfolioPDF()
        pdf.set_auto_page_break(auto=True,margin=14)
        pdf.set_margins(10,20,10)

        # Portada
        pdf.add_page()
        pdf.set_fill_color(*pdf.C_BG); pdf.rect(0,0,210,297,'F')
        pdf.set_fill_color(*pdf.C_ACCENT); pdf.rect(0,85,210,2,'F'); pdf.rect(0,205,210,2,'F')
        pdf.set_y(100); pdf.set_font('Helvetica','B',28)
        pdf.set_text_color(*pdf.C_ACCENT); pdf.cell(0,14,'PORTFOLIO ANALYSIS',align='C',ln=True)
        pdf.set_text_color(255,255,255); pdf.cell(0,14,'INSTITUTIONAL v3',align='C',ln=True)
        pdf.ln(6); pdf.set_font('Helvetica','',11); pdf.set_text_color(*pdf.C_MUTED)
        pdf.cell(0,6,_t('Markowitz + Black-Litterman + Risk Parity + CVaR + Stress + FF + MC'),align='C',ln=True)
        pdf.ln(8); pdf.set_font('Helvetica','B',9); pdf.set_text_color(*pdf.C_GOLD)
        pdf.cell(0,5,_t(f'Activos: {" | ".join(TICKERS)}'),align='C',ln=True)
        pdf.ln(3); pdf.set_font('Helvetica','',8); pdf.set_text_color(*pdf.C_MUTED)
        pdf.cell(0,5,_t(f'NAV: ${NAV:,.0f} | RF: {RF_A:.2%} | LW Shrinkage: {shrinkage:.4f}'),align='C',ln=True)
        pdf.cell(0,5,_t(f'{FECHA_INI} a {FECHA_FIN} | {datetime.today().strftime("%d/%m/%Y %H:%M")}'),align='C',ln=True)

        # Resumen ejecutivo
        pdf.add_page(); pdf.sec('1. Resumen Ejecutivo')
        pdf.set_font('Helvetica','',9); pdf.set_text_color(*pdf.C_TEXT)
        pdf.multi_cell(0,5,_t(f'Analisis cuantitativo institucional de {N} activos. '
            f'Covarianza estimada con Ledoit-Wolf (shrinkage={shrinkage:.4f}). '
            f'Cuatro portafolios: PMV, PMS (Markowitz), Risk Parity y Black-Litterman. '
            f'Riesgo: VaR parametrico, historico y Monte Carlo al 95/99%. CVaR/Expected Shortfall. '
            f'Stress testing en {len(PERIODOS_STRESS)} periodos historicos. '
            f'Rolling analysis 1Y/2Y/3Y. Factores Fama-French proxy. Regimenes de mercado K-Means. '
            f'Monte Carlo GBM con {N_SIMS:,} simulaciones. Trade list con costo estimado.'))
        pdf.ln(8)
        pdf.set_font('Helvetica','B',10); pdf.set_text_color(*pdf.C_ACCENT)
        pdf.cell(0,6,'KPIs — Portafolio Max Sharpe (PMS)',ln=True); pdf.ln(3)
        pdf.kpi_row([
            (f"{pms['rendimiento']:.2%}",'Rend. Anual','PMS esperado'),
            (f"{pms['riesgo']:.2%}",'Sigma Anual','PMS anualizado'),
            (f"{pms['sharpe']:.3f}",'Ratio Sharpe','PMS optimizado'),
            (f"{riesgo_pms['CVaR 99% (ES)']:.2%}",'CVaR 99%','Expected Shortfall'),
            (f"{max_dd_pms:.2%}",'Max Drawdown','Peor caida historica'),
            (f"{calmar_pms:.2f}",'Calmar Ratio','Rend/MaxDD'),
        ])

        # Estadísticos
        pdf.sec('2. Estadisticos por Activo')
        pdf.tabla(stats[['Rend_Anual','Std_Anual','Sharpe','Skewness','Kurtosis']].round(4))

        # Portafolios
        pdf.add_page(); pdf.sec('3. Portafolios Optimizados')
        for p,lbl in [(pmv,'Min Varianza PMV'),(pms,'Max Sharpe PMS'),
                      (prp,'Risk Parity'),(pbl,'Black-Litterman')]:
            pdf.set_font('Helvetica','B',9); pdf.set_text_color(*pdf.C_GOLD)
            pdf.cell(0,6,lbl,ln=True)
            pdf.set_font('Helvetica','',8); pdf.set_text_color(*pdf.C_TEXT)
            pdf.cell(0,5,_t(f'Rend:{p["rendimiento"]:.2%} sigma:{p["riesgo"]:.2%} Sharpe:{p["sharpe"]:.3f}'),ln=True)
            pdf.ln(1)
            dp=pd.DataFrame({'Ticker':TICKERS,'Peso':[f'{w:.2%}' for w in p['pesos']]}).set_index('Ticker')
            pdf.tabla(dp); pdf.ln(3)

        # Imagenes
        for f,tit in [('frontera.png','4. Frontera Eficiente + CML'),
                      ('corr.png','5. Matriz de Correlacion (Ledoit-Wolf)'),
                      ('dd.png','6. Drawdown — Underwater Chart'),
                      ('stress.png','7. Stress Testing Historico'),
                      ('rolling.png','8. Rolling Sharpe 1Y/2Y/3Y'),
                      ('ff.png','9. Factores Fama-French'),
                      ('mc.png','10. Monte Carlo GBM')]:
            if IMGS_OK and os.path.exists(f'{IMG_DIR}/{f}'):
                pdf.add_page(); pdf.sec(tit)
                pdf.img_safe(f'{IMG_DIR}/{f}',x=10,w=190)

        # Riesgo
        pdf.add_page(); pdf.sec('11. Riesgo — VaR / CVaR / Expected Shortfall')
        pdf.tabla(df_riesgo.applymap(lambda x: f'{x:.4f}' if isinstance(x,float) else str(x)))

        # Stress
        pdf.add_page(); pdf.sec('12. Stress Testing — Detalle')
        pdf.tabla(df_stress.reset_index().set_index('Periodo' if 'Periodo' in df_stress.reset_index().columns else df_stress.reset_index().columns[0]).round(4))

        # Drawdown
        pdf.add_page(); pdf.sec('13. Drawdown Analysis')
        dd_data=pd.DataFrame({'Max DD':[max_dd_pmv,max_dd_pms],'Calmar':[calmar_pmv,calmar_pms],
                              'Rec. Dias':[rec_pmv or 'No rec.',rec_pms or 'No rec.']},
                             index=['PMV','PMS'])
        pdf.tabla(dd_data)

        # MC
        pdf.add_page(); pdf.sec('14. Monte Carlo GBM — Resumen')
        mc_tab=pd.DataFrame(mc_rows).set_index(['Port','Anos'])
        pdf.tabla(mc_tab.round(4))

        # Rebalanceo
        pdf.add_page(); pdf.sec('15. Trade List — Rebalanceo hacia PMS')
        reb_show=df_reb_pms[['Peso Actual','Peso Objetivo','Gap','Direccion' if 'Direccion' in df_reb_pms.columns else 'Dirección','USD Trade','Comision Est.' if 'Comision Est.' in df_reb_pms.columns else 'Comisión Est.']].copy()
        reb_show.columns=[_t(c) for c in reb_show.columns]
        pdf.tabla(reb_show.round(4))
        pdf.set_font('Helvetica','',8); pdf.set_text_color(*pdf.C_MUTED)
        pdf.cell(0,5,_t(f'Turnover: ${df_reb_pms["USD Trade"].sum():,.0f} ({df_reb_pms["USD Trade"].sum()/NAV:.1%} del NAV) | Costo est.: ${df_reb_pms["Comision Est."].sum() if "Comision Est." in df_reb_pms.columns else df_reb_pms["Comisión Est."].sum():,.0f}'),ln=True)

        # Señales técnicas
        pdf.add_page(); pdf.sec('16. Senales Tecnicas (90 dias)')
        for tk,ind2 in inds.items():
            pdf.set_font('Helvetica','B',9); pdf.set_text_color(*pdf.C_GOLD)
            pdf.cell(0,6,tk,ln=True)
            pdf.set_font('Helvetica','',8); pdf.set_text_color(*pdf.C_TEXT)
            for s in ind2['señales']: pdf.cell(0,5,_t(f'  {s}'),ln=True)
            pdf.ln(2)

        # Notas metodológicas
        pdf.add_page(); pdf.sec('17. Notas Metodologicas')
        notas=[
            ('Rendimientos','Logaritmicos ln(Pt/Pt-1). Anualizacion: mu*F, sigma*sqrt(F).'),
            ('Ledoit-Wolf',f'Shrinkage={shrinkage:.4f}. Reduce error estimacion matriz covarianza.'),
            ('Markowitz',f'SLSQP. Pesos [{PESO_MIN:.0%},{PESO_MAX:.0%}]. Sin cortos.'),
            ('Black-Litterman',f'tau={BL_TAU}, confianza={BL_CONF:.0f}%. Equilibrio+views PM.'),
            ('Risk Parity','Equal Risk Contribution. Minimiza dispersion contribuciones de riesgo.'),
            ('CVaR/ES','Expected Shortfall: perdida esperada dado que supera el VaR.'),
            ('Stress','7 periodos historicos de crisis. Retorno total y max DD real.'),
            ('Drawdown','Calmar=Rend_anual/|MaxDD|. Tiempo recuperacion desde trough.'),
            ('Fama-French','Proxy ETF: Mkt=SPY, SMB=IWM-SPY, HML=IVE-IWF, MOM=MTUM-USMV.'),
            ('Regimenes','K-Means sobre ret/vol rolling 21d. 3 estados: Bull/Lateral/Bear.'),
            ('Monte Carlo',f'GBM, {N_SIMS:,} sims. S_t=S_0*exp((mu-0.5*sig^2)*t+sig*sqrt(t)*Z).'),
            ('Rebalanceo','Delta acciones = (NAV*w_obj/precio) - posicion_actual. Costo=delta*precio*comision.'),
        ]
        pdf.set_font('Helvetica','',8)
        for con,desc in notas:
            pdf.set_text_color(*pdf.C_ACCENT); pdf.set_font('Helvetica','B',8)
            pdf.cell(38,5,_t(con),border='B')
            pdf.set_text_color(*pdf.C_TEXT); pdf.set_font('Helvetica','',8)
            pdf.multi_cell(0,5,_t(desc),border='B'); pdf.ln(1)

        PDF_PATH='portfolio_report_v3.pdf'
        pdf.output(PDF_PATH)
        print(f'✅ PDF: {PDF_PATH} ({os.path.getsize(PDF_PATH)//1024} KB)')

        # Descargas Colab
        try:
            from google.colab import files
            for f in [HTML_PATH,EXCEL_PATH,PDF_PATH]: files.download(f)
        except ImportError: pass

        print(f'\n{'='*62}')
        print('  ANALISIS INSTITUCIONAL COMPLETO')
        print(f'  HTML  : {HTML_PATH}')
        print(f'  Excel : {EXCEL_PATH}  ({wb.sheetnames})')
        print(f'  PDF   : {PDF_PATH}')
        print(f'{'='*62}')
        w_status.value='<span style="color:#3fb950">✅ Completo. 3 archivos listos.</span>'


w_run.on_click(ejecutar_analisis)
print('✅ Pipeline institucional v3 listo.')
print('   Presiona [ ▶ EJECUTAR ANÁLISIS COMPLETO ] en el panel de arriba.')